In [2]:
!pip install https://github.com/omwaman1/dlib_wheel_for_python/releases/download/dlib-python3.13.5/dlib-20.0.0-cp313-cp313-win_amd64.whl

     ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
     ---------------------------------------- 2.9/2.9 MB 38.7 MB/s  0:00:00


In [3]:
!pip install face-recognition

  Using cached face_recognition-1.3.0-py2.py3-none-any.whl.metadata (21 kB)
  Using cached face_recognition_models-0.3.0-py2.py3-none-any.whl
Using cached face_recognition-1.3.0-py2.py3-none-any.whl (15 kB)

   ---------------------------------------- 0/2 [face-recognition-models]
   ---------------------------------------- 0/2 [face-recognition-models]
   ---------------------------------------- 0/2 [face-recognition-models]
   ---------------------------------------- 2/2 [face-recognition]



In [5]:
import os
print("Current folder:", os.getcwd())
print("\nFolders found here:", os.listdir("."))

Current folder: C:\Users\grana\face_recognition_attendence_system

Folders found here: ['.ipynb_checkpoints', 'download_dataset.ipynb', 'encode_faces.ipynb', 'encodings.pkl', 'known_faces']


In [6]:
import os

dataset_path = "known_faces"

if not os.path.exists(dataset_path):
    print("❌ known_faces folder NOT FOUND in:", os.getcwd())
else:
    print("✅ known_faces folder found!")
    people = os.listdir(dataset_path)
    print(f"People folders inside: {len(people)}")
    for person in people[:5]:  # Show first 5
        person_path = os.path.join(dataset_path, person)
        images = os.listdir(person_path)
        print(f"  {person} → {len(images)} images")

✅ known_faces folder found!
People folders inside: 62
  Alejandro_Toledo → 39 images
  Alvaro_Uribe → 35 images
  Amelie_Mauresmo → 21 images
  Andre_Agassi → 36 images
  Angelina_Jolie → 20 images


In [7]:
import face_recognition
import os
import pickle

known_encodings = []
known_names = []

dataset_path = "known_faces"
total = 0
skipped = 0

for person_name in os.listdir(dataset_path):
    person_folder = os.path.join(dataset_path, person_name)
    if not os.path.isdir(person_folder):
        continue

    for image_file in os.listdir(person_folder):
        img_path = os.path.join(person_folder, image_file)
        image = face_recognition.load_image_file(img_path)

        # ✅ Fix: upsample=2 helps detect small/cropped LFW images
        face_locations = face_recognition.face_locations(image, number_of_times_to_upsample=2)
        encodings = face_recognition.face_encodings(image, known_face_locations=face_locations)

        if encodings:
            known_encodings.append(encodings[0])
            known_names.append(person_name)
            total += 1
        else:
            skipped += 1

    print(f"✅ Encoded: {person_name}")

# Save to file
with open("encodings.pkl", "wb") as f:
    pickle.dump({"encodings": known_encodings, "names": known_names}, f)

print(f"\n🎉 Done! {total} encodings saved.")
print(f"⚠️ Skipped {skipped} images (face not detectable).")

✅ Encoded: Alejandro_Toledo
✅ Encoded: Alvaro_Uribe
✅ Encoded: Amelie_Mauresmo
✅ Encoded: Andre_Agassi
✅ Encoded: Angelina_Jolie
✅ Encoded: Ariel_Sharon
✅ Encoded: Arnold_Schwarzenegger
✅ Encoded: Atal_Bihari_Vajpayee
✅ Encoded: Bill_Clinton
✅ Encoded: Carlos_Menem
✅ Encoded: Colin_Powell
✅ Encoded: David_Beckham
✅ Encoded: Donald_Rumsfeld
✅ Encoded: George_Robertson
✅ Encoded: George_W_Bush
✅ Encoded: Gerhard_Schroeder
✅ Encoded: Gloria_Macapagal_Arroyo
✅ Encoded: Gray_Davis
✅ Encoded: Guillermo_Coria
✅ Encoded: Hamid_Karzai
✅ Encoded: Hans_Blix
✅ Encoded: Hugo_Chavez
✅ Encoded: Igor_Ivanov
✅ Encoded: Jack_Straw
✅ Encoded: Jacques_Chirac
✅ Encoded: Jean_Chretien
✅ Encoded: Jennifer_Aniston
✅ Encoded: Jennifer_Capriati
✅ Encoded: Jennifer_Lopez
✅ Encoded: Jeremy_Greenstock
✅ Encoded: Jiang_Zemin
✅ Encoded: John_Ashcroft
✅ Encoded: John_Negroponte
✅ Encoded: Jose_Maria_Aznar
✅ Encoded: Juan_Carlos_Ferrero
✅ Encoded: Junichiro_Koizumi
✅ Encoded: Kofi_Annan
✅ Encoded: Laura_Bush
✅ Encoded

In [9]:
import face_recognition
import pickle
from sklearn.datasets import fetch_lfw_people

# Reload dataset (instant - already cached on your PC)
lfw = fetch_lfw_people(min_faces_per_person=20, resize=1.0, color=True)

known_encodings = []
known_names = []
total = 0
skipped = 0

for idx, (image, label) in enumerate(zip(lfw.images, lfw.target)):
    name = lfw.target_names[label].replace(' ', '_')

    # ✅ KEY FIX: multiply by 255 since values are 0.0-1.0
    img_uint8 = (image * 255).astype('uint8')

    face_locs = face_recognition.face_locations(img_uint8, number_of_times_to_upsample=2)
    encodings = face_recognition.face_encodings(img_uint8, known_face_locations=face_locs)

    if encodings:
        known_encodings.append(encodings[0])
        known_names.append(name)
        total += 1
    else:
        skipped += 1

    # Progress every 100 images
    if idx % 100 == 0:
        print(f"⏳ {idx}/{len(lfw.images)} processed — {total} encoded so far...")

# Save encodings
with open("encodings.pkl", "wb") as f:
    pickle.dump({"encodings": known_encodings, "names": known_names}, f)

print(f"\n🎉 Done! {total} encodings saved.")
print(f"⚠️  Skipped {skipped} images (no face detected).")

⏳ 0/3023 processed — 1 encoded so far...
⏳ 100/3023 processed — 91 encoded so far...
⏳ 200/3023 processed — 180 encoded so far...
⏳ 300/3023 processed — 267 encoded so far...
⏳ 400/3023 processed — 359 encoded so far...
⏳ 500/3023 processed — 454 encoded so far...
⏳ 600/3023 processed — 547 encoded so far...
⏳ 700/3023 processed — 641 encoded so far...
⏳ 800/3023 processed — 729 encoded so far...
⏳ 900/3023 processed — 819 encoded so far...
⏳ 1000/3023 processed — 908 encoded so far...
⏳ 1100/3023 processed — 998 encoded so far...
⏳ 1200/3023 processed — 1088 encoded so far...
⏳ 1300/3023 processed — 1184 encoded so far...
⏳ 1400/3023 processed — 1275 encoded so far...
⏳ 1500/3023 processed — 1366 encoded so far...
⏳ 1600/3023 processed — 1453 encoded so far...
⏳ 1700/3023 processed — 1548 encoded so far...
⏳ 1800/3023 processed — 1637 encoded so far...
⏳ 1900/3023 processed — 1731 encoded so far...
⏳ 2000/3023 processed — 1823 encoded so far...
⏳ 2100/3023 processed — 1913 encoded so 